In [ ]:
import pandas as pd
import pymysql
from pymongo import MongoClient
import psycopg2
import matplotlib.pyplot as plt
import time as tm
from folium import plugins
import json
import urllib.request
from math import radians, cos, sin, asin, sqrt
import numpy as np
import datetime
from datetime import datetime
from tqdm import tqdm
from dask import delayed
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:200% !important; }</style>"))


In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py




db = client.get_database('telematics_db')

db2 = client2.get_database('telematics_db')





def gettime(x):
    try:
        return pd.Timestamp(x).timestamp()
    except:
        return None

In [ ]:
def haversine(L1,L2):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees)
    """

    lon1=L1[0]
    lat1=L1[1]
    lon2=L2[0]
    lat2=L2[1]
    #print(lon1,lat1,lon2,lat2)
    # convert decimal degrees to radians
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])

    # haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 6371000 # Radius of earth in kilometers. Use 3956 for miles
    return c * r

In [ ]:
csvwriter = open('jump_delay_gps_gsm.csv','a')



def no_info(vehicleId):
    print('>>>>>>><<<<<<<')
    try:
        query={'vehicleId':vehicleId,'time':{'$gte':tm.time()-7*86400}}
        data1=pd.DataFrame(list(db2.gpsHistory.find(query)))       
        print(' MONGO AA GAYAA, WITH LENGTH ' + str(len(data1)) )
        
        
        #url=GPS_API_BASE+"/gps/vehicles?vehicleIds={}&fromTime={}&toTime={}&sortOrder=&timeCondition=".format(vehicleId,from_tm,to_tm)
        #data=json.load(urllib.request.urlopen(url))
        #data1=data['data'][str(vehicleId)]
          
        
        
    except:
        
        print('mongo error')
    
    if len(data1)>2:
        
        
        
        try:
            data1.sort_values('time',ascending=True,inplace=True)
            data1['loc2']=data1.location.shift(1)
            data1=data1[1:]
            
            loc_list=data1[['location','loc2']].values.tolist()

            dis=[]
            for loc in loc_list:
                y=haversine(loc[0],loc[1])
                dis.append(y)
            
            data1['distance']=dis
            
            jump_percent_distance=data1[data1['distance']>500]['distance'].sum()/data1['distance'].sum()
            total_pings=len(data1)
            
            
            data1.created=data1.created.apply(gettime)
            
            data1['server_delay']=data1.created-data1.time
            global distace_values_500
            distace_values_500=list(data1[data1['distance']>500]['distance'].describe(percentiles=[0.75,0.85,0.90,0.95,0.99]).values[[0,1,4,5,6,7,8,9,10]])
            distace_values_500=[0 if dv is '' else dv for dv in distace_values_500]
            delay_values_5min=list(data1[data1['server_delay']>300]['server_delay'].describe(percentiles=[0.75,0.85,0.90,0.95,0.99]).values[[0,1,4,5,6,7,8,9,10]])
            delay_values_5min=[0 if v is None else v for v in delay_values_5min]
        
            try:
                gsm_values=list(data1['gsmPct'].describe(percentiles=[0.01,0.05,0.2,0.3,0.4,0.6]).values[[1,4,5,6,7,8,9,10]])
                
            except:
                gsm_values=[None]*8
                
            try:
                gps_values=list(data1['usedSat'].describe(percentiles=[0.01,0.05,0.2,0.3,0.4,0.6]).values[[1,4,5,6,7,8,9,10]])
                
            except:
                gps_values=[None]*8
            global data2
            
            data2=pd.DataFrame([[vehicleId,total_pings,jump_percent_distance]+distace_values_500+delay_values_5min+gsm_values+gps_values],
                               columns=['vehicleId','total_pings','jump_percent_distance',
                                        'jump_count','jump_mean','jump_50%', 'jump_75%', 'jump_85%', 'jump_90%', 'jump_95%', 'jump_99%','jump_max',
                                       'delay_count','delay_mean','delay_50%', 'delay_75%', 'delay_85%', 'delay_90%', 'delay_95%', 'delay_99%','delay_max',
                                       'gsm_mean','gsm_1%', 'gsm_5%', 'gsm_20%',  'gsm_30%', 'gsm_40%','gsm_50%','gsm_60%',
                                       'gps_mean','gps_1%', 'gps_5%', 'gps_20%',  'gps_30%', 'gps_40%','gps_50%','gps_60%'])
            
            
            
            
            data2[['jump_count','jump_mean','jump_50%', 'jump_75%', 'jump_85%', 'jump_90%', 'jump_95%', 'jump_99%','jump_max']]=data2[['jump_count','jump_mean','jump_50%', 'jump_75%', 'jump_85%', 'jump_90%', 'jump_95%', 'jump_99%','jump_max']].fillna(0)
            row = data2.to_csv(header=False)
            print(row)
            csvwriter.write(row)
            csvwriter.flush()


        except Exception as e:
            print(e)
    else:
        print("NO DATA FROM LAST 7 DAYS")
    
            
    return None



In [ ]:
vehicles=pd.read_sql("select vehicle_id from operator_vehicles where deboarding_time is null",con=mysql_vds)
print('1')
#removing wire cuts

device_history=pd.DataFrame(list(db.deviceHistory.find({'toTime':{'$gte':tm.time()-9*86400},'suggestion':'Battery disconnected'})))
device_hist2=pd.DataFrame(list(db.deviceHistory.find({'toTime':None,'suggestion':'Battery disconnected'})))
device_history=pd.concat([device_history,device_hist2])
print('2')
try:
    back_log=pd.read_csv("jump_delay_gps_gsm.csv")

    set1=set(back_log['vehicle_id'])

except:
    set1=set()

print('3')
set2=set(device_history.vehicleId)
set3=set(vehicles['vehicle_id'])
print('4')
vehicles_list=set3-set1-set2
vehicles_list=[100001]


%%time
complete_data=pd.DataFrame()
analysed_data=pd.DataFrame()
count=len(pd.read_csv("diatance_in_no_info.csv"))
for i in device_history:
    _, analysed_data, complete_data =no_info(i, analysed_data, complete_data)

In [ ]:

from multiprocessing.pool import ThreadPool as Pool
pool = Pool() 

pool.map(no_info, vehicles_list)

pool.close()
pool.join()

# ANALYSIS AFTER DOWNLOADING CSV FROM EC2

In [ ]:
data=pd.read_csv("jump_delay_gps_gsm.csv")

In [ ]:
data.drop_duplicates('vehicle_id',inplace=True)


In [ ]:
data=data[~data['vehicle_id'].isin(set2)]

In [ ]:
data.jump_percent_distance=data.jump_percent_distance*100

In [ ]:
del data['len','total_time','min_expected_pings','suggestion','date_of_no_info']

In [ ]:
data['jump_percent_distance']=data['total_jump']/data['total_distance']

In [ ]:
data['jump_percent_distance']=data['jump_percent_distance']*100

In [ ]:
vehicle_details=pd.read_sql("select vehicle_id,model_name,sub_type_installation from vehicle_details",con=postgres)

In [ ]:
vehicle_details.head()

In [ ]:
data=pd.merge(data,vehicle_details,left_on='vehicle_id',right_on='vehicle_id',how='left')

In [ ]:
device_models=pd.read_sql("select * from device_models",con=mysql_cn)

our_devices=['GT06N','WETRACK2','WETRACK-LITE','C100','TL1000','TL250','WETRACK800','AW100','WP30L','GRLAIS','AT50','WP30L','FMB920','FMB207','FMB150','FMB001','LC100_SOS','WETRACK2_SOS','AT4','ATLANTA_OBD','BHARAT101','TRANSYNC_AIS_140','GAGAN']                           


our_device_list=[]

for i, row in device_models.iterrows():
    if row['name'] in our_devices:
        our_device_list.append('yes')
    else:
        our_device_list.append('no')
device_models['our_device']=our_device_list

In [ ]:
data=pd.merge(data,device_models,left_on='model_name',right_on='name',how='left')

In [ ]:
data[data['sub_type_installation']=='DEVICE_AND_SIM_CHANGE'].sub_type_installation.value_counts()

In [ ]:
data[data['name']=='GAGAN']

In [ ]:
del data['l']

#### data['jump_per_ping'].describe

In [ ]:
plt.hist(data[data['sub_type_installation']=='EXTERNAL_API']['total_pings'],bins=100,label='device_and_sim_change')

In [ ]:
data['jump_per_ping']=data['total_jump']/data['jump_pings']

In [ ]:
data['total_jump']=data['total_jump']/1000

In [ ]:
data[data['our_device']=='yes']['jump_per_ping'].describe()

In [ ]:
data['our_device'].fillna('API',inplace=True)

In [ ]:
complete_data_list=[]
for our in ['yes','no','API']:
    for installation_type in list(data.sub_type_installation.unique()):
        for model in list(data.model_name.unique()):
            current_data_set=data[data['our_device']==our][data['sub_type_installation']==installation_type][data['model_name']==model]
            row_of_compiled=[our,installation_type,model,len(current_data_set)]
            for column in list(data.columns)[1:37]:
                val=current_data_set[column].mean()
                row_of_compiled=row_of_compiled+[val]
                
                
            complete_data_list.append(row_of_compiled)

            
final_texts=['our_device','installation_type','model_name','count']+['total_pings',
 'jump_percent_distance',
 'jump_count',
 'jump_mean',
 'jump_50%',
 'jump_75%',
 'jump_85%',
 'jump_90%',
 'jump_95%',
 'jump_99%',
 'jump_max',
 'delay_count',
 'delay_mean',
 'delay_50%',
 'delay_75%',
 'delay_85%',
 'delay_90%',
 'delay_95%',
 'delay_99%',
 'delay_max',
 'gsm_mean',
 'gsm_1%',
 'gsm_5%',
 'gsm_20%',
 'gsm_30%',
 'gsm_40%',
 'gsm_50%',
 'gsm_60%',
 'gps_mean',
 'gps_1%',
 'gps_5%',
 'gps_20%',
 'gps_30%',
 'gps_40%',
 'gps_50%',
 'gps_60%']

ultimate_data=pd.DataFrame(complete_data_list,columns=final_texts)


In [ ]:
ultimate_data

In [ ]:
ultimate_data=ultimate_data[ultimate_data['jump_percent_distance'].notnull()]

In [ ]:
ultimate_data.reset_index(inplace=True)

In [ ]:
len(ultimate_data.model_name.unique())

In [ ]:
ultimate_data.to_csv("final_ultimate_data.csv")

In [ ]:
list(data.columns)[1:37]



In [ ]:
a=data.jump_percent_distance.describe(percentiles=[0.75,0.85,0.90,0.95,0.975,0.99]).keys()[[1,5,6,7,8,9,10]]

In [ ]:
a

In [ ]:
data.jump_percent_distance.describe(percentiles=[0.75,0.85,0.90,0.95,0.975,0.99])

In [ ]:
ultimate_data.to_csv("ultimate_data.csv")

In [ ]:
data=pd.read_csv("ultimate_data.csv")

In [ ]:
data[~data['vehicleId'].isin(set2)]

In [ ]:
data.head()

In [ ]:
ultimate_data.total_pings_mean